In [1]:
import pandas as pd

In [2]:
# Load the raw EDFacts LEA file
input_path = "../data/raw/SY1018_FS150_FS151_DG695_DG696_LEA.csv"

In [3]:
df = pd.read_csv(input_path, low_memory=False)
df.head()

,School Year,State,NCES LEA ID,LEA,School,NCES SCH ID,Data Group,Data Description,Value,Numerator,Denominator,Population,Subgroup,Characteristics,Age/Grade,Academic Subject,Outcome,Program Type
0,2010-2011,ALABAMA,100005,Albertville City,NaN,NaN,695|696,Four-Year Adjusted-Cohort Graduation Rates,80,NaN,252,All Students,All Students,NaN,NaN,NaN,NaN,NaN
1,2010-2011,ALABAMA,100005,Albertville City,NaN,NaN,695|696,Four-Year Adjusted-Cohort Graduation Rates,PS,NaN,3,All Students,Children with disabilities,NaN,NaN,NaN,NaN,NaN
2,2010-2011,ALABAMA,100005,Albertville City,NaN,NaN,695|696,Four-Year Adjusted-Cohort Graduation Rates,65-69,NaN,121,All Students,Economically Disadvantaged,NaN,NaN,NaN,NaN,NaN
3,2010-2011,ALABAMA,100005,Albertville City,NaN,NaN,695|696,Four-Year Adjusted-Cohort Graduation Rates,LT50,NaN,10,All Students,English Learner,NaN,NaN,NaN,NaN,NaN
4,2010-2011,ALABAMA,100005,Albertville City,NaN,NaN,695|696,Four-Year Adjusted-Cohort Graduation Rates,GE50,NaN,12,All Students,Black or African American,NaN,NaN,NaN,NaN,NaN


In [4]:
print('Raw Shape:', df.shape)

Raw Shape: (706595, 18)


In [5]:
df.dtypes

School Year          object
State                object
NCES LEA ID           int64
LEA                  object
School              float64
NCES SCH ID         float64
Data Group           object
Data Description     object
Value                object
Numerator           float64
Denominator           int64
Population           object
Subgroup             object
Characteristics     float64
Age/Grade           float64
Academic Subject    float64
Outcome             float64
Program Type        float64
dtype: object

In [6]:
# Filter data to single year and single population
df = df[
    (df['Subgroup'] == 'All Students') &
    (df['School Year'] == '2017-2018') &
    (pd.to_numeric(df['Denominator'], errors='coerce') >= 100)
].copy()

In [7]:
df['Value'].unique()

array(['94', '91', '96', 'GE95', '90-94', '97', '89', '75-79', '95', '93',
       '88', '77', '79', '85-89', '83', '92', '82', '87', '80-84', '90',
       '84', '86', '81', '98', '61', '60-64', '80', '85', '35-39', '78',
       '65-69', '55', '20-24', '32', '23', '52', '53', '50-54', '27',
       '19', '18', '46', '25-29', '30-34', '28', '21', '45-49', '36',
       '40-44', '14-Oct', '12', '71', 'LE1', '76', '72', '70-74', '74',
       'GE99', '69', '75', '58', '73', '29', '43', '31', '22', '15-19',
       '48', 'LE5', '45', '25', '64', '55-59', '70', '56', '60', '68',
       '3', '67', '66', '7', '13', '8', '50', '17', '65', '38', '16',
       '51', '2', '40', '9-Jun', '59', '34', '37', '30', '54', '47', '62',
       '41', '63', '57', '24', '6', '49', '42', '26', '35'], dtype=object)

## Handling suppressed graduation rate values

EDFacts suppresses exact graduation rates in two cases: when a subgroup is small enough that an exact percentage could identify individual 
students, and when a rate is very close to 0% or 100%, since a near-extreme rate also narrows down individual outcomes. 
Suppressed values appear as ranges (e.g. "90-94"), thresholds (e.g. "GE95", "LE5"), or a fully masked code ("PS").

Rather than drop these rows, which would have removed ~41% of the sample and systematically excluded the highest-performing districts, I imputed a single numeric estimate for each suppressed code:
- Range codes: midpoint of the range (e.g. "90-94" -> 92)
- Threshold codes: the boundary value itself (e.g. "GE95" -> 95, "LE5" -> 5)

Two entries ("14-Oct", "9-Jun") were corrected first: these were originally range codes ("10-14", "6-9") that Excel auto-converted to dates at some point.

Limitation: these are estimates, not measured values. Boundary-value imputation for GE/LE codes likely slightly understates high rates and 
overstates low rates, since it doesn't split the difference toward the true ceiling/floor.

In [8]:
# Correct Excel errors and convert 'grad_rate' to numeric
def impute_code(code):
    code = str(code)

    # Fix Excel's date corruption first
    excel_fixes = {"14-Oct": "10-14", "9-Jun": "6-9"}
    if code in excel_fixes:
        code = excel_fixes[code]

    # "GE95" -> boundary value, e.g. 95
    if code.startswith("GE"):
        return pd.to_numeric(code.split('GE')[1])

    # "LE5" -> boundary value, e.g. 5
    if code.startswith("LE"):
        return pd.to_numeric(code.split('LE')[1])

    # "90-94" -> mean of the two numbers
    if "-" in code:
        low, high = code.split("-")
        low = pd.to_numeric(low)
        high = pd.to_numeric(high)
        return (low + high) / 2

    # Return clean number
    return pd.to_numeric(code)

In [9]:
# Rename columns
df = df.rename(columns={
    'NCES LEA ID': 'LEAID'
})

In [10]:
# Map impute_code to 'grad_rate' and filter for values
df['grad_rate'] = df['Value'].apply(impute_code)
df = df[(df['grad_rate'] >= 0) &
    (df['grad_rate'] <= 100)]
print('After filtering', df.shape)
print('Unique LEAID', df['LEAID'].nunique())
print('Sum of NaN:', df['grad_rate'].isna().sum())

After filtering (6113, 19)
Unique LEAID 6113
Sum of NaN: 0


In [11]:
# Check duplicates were dropped
dupes = df['LEAID'].value_counts()
print('Max rows per LEAID:', dupes.max())

Max rows per LEAID: 1


In [12]:
# Drop duplicates
df_lea = df[['LEAID', 'grad_rate']].drop_duplicates()

In [13]:
# Save processed .csv file
output_path = '../data/processed/edfacts_lea_graduation_2017_2018_v2.csv'
df_lea.to_csv(output_path, index=False)
print(f'Saved to {output_path}')
print(df_lea.head())

Saved to ../data/processed/edfacts_lea_graduation_2017_2018_v2.csv
         LEAID  grad_rate
602244  100005       94.0
602254  100006       91.0
602265  100007       94.0
602276  100008       96.0
602288  100011       95.0
